# Modeling Train  Tracking Registering the models 

In [87]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

In [88]:
import pandas as pd
from typing import List, Tuple

def load_and_combine_months(file_paths: List[str]) -> pd.DataFrame:
    """
    Load multiple monthly parquet files and combine them into one DataFrame.
    Adds a 'month' column for time-based splitting.
    """
    dfs = []
    
    for path in file_paths:
        print(f"Loading: {path}")
        df = pd.read_parquet(path)
        
        # استخراج ماه از نام فایل (فرض: نام فایل شامل تاریخ است)
        # مثال: yellow_tripdata_2026-01.parquet → month = 1
        month = int(path.split('-')[-1].split('.')[0].split('_')[0])

        df['month'] = month
        
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"Total combined data shape: {combined.shape}")
    return combined


def split_train_validation_by_month(
    df: pd.DataFrame, 
    train_months: List[int], 
    val_month: int
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split data into Train and Validation based on month.
    """
    train_df = df[df['month'].isin(train_months)].copy()
    val_df = df[df['month'] == val_month].copy()
    
    print(f"Train shape: {train_df.shape} | Months: {train_months}")
    print(f"Validation shape: {val_df.shape} | Month: {val_month}")
    
    return train_df, val_df

In [89]:
# مسیر فایل‌ها
file_paths = [
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-01_processed.parquet",
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-02_processed.parquet",
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-03_processed.parquet",
]

# بارگذاری و ترکیب
df = load_and_combine_months(file_paths)

# تقسیم زمانی
train, val = split_train_validation_by_month(
    df=df,
    train_months=[1, 2],   # ماه ۱ و ۲ برای Train
    val_month=3            # ماه ۳ برای Validation
)


Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-01_processed.parquet
Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-02_processed.parquet
Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-03_processed.parquet
Total combined data shape: (7748178, 28)
Train shape: (4838269, 28) | Months: [1, 2]
Validation shape: (2909909, 28) | Month: 3


In [90]:
train_df = train.sample(n=5000, random_state = 42).reset_index(drop=True)
val_df = val.sample(n=500, random_state = 42).reset_index(drop=True)

In [11]:
def select_features(df: pd.DataFrame):
    """Select features and define categorical vs numerical."""
    categorical = ['hour_category','PU_DO']           # فقط این را وکتورایز می‌کنیم (تعداد دسته کم است)
    
    numerical = [
        'trip_distance',
        'PULocationID',
        'DOLocationID',
        'pickup_dayofweek',
        'pickup_hour',
        'is_rush_hour',
        'is_weekend'
    ]
    target = 'duration'

    
    df = df[categorical + numerical + [target]].copy()
    return df, categorical, numerical, target


def vectorize_features(train_df, val_df, categorical, numerical):
    """Convert features using DictVectorizer."""
    print("Vectorizing features...")

    dv = DictVectorizer()

    train_dicts = train_df[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = val_df[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    return X_train, X_val, dv


In [12]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment('nyc_duration_prediction')

Tracking URI: http://127.0.0.1:5000


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1785237891492, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785237891492, lifecycle_stage='active', name='nyc_duration_prediction', tags={}, trace_location=None, workspace='default'>

In [73]:
import mlflow
import mlflow.sklearn
import joblib
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction import DictVectorizer
from pathlib import Path

with mlflow.start_run(run_name="rf_target_encoding_v7"):

    # ========================
    # انتخاب فیچرها
    # ========================
    train_df, categorical, numerical, target = select_features(train_df)
    val_df, _, _, _ = select_features(val_df)

    # ========================
    # Target Encoding برای PU_DO (حرفه‌ای)
    # ========================
    target_encoder = TargetEncoder(
        cols=['PU_DO'], 
        smoothing=15, 
        min_samples_leaf=30
    )
    
    # فقط روی داده Train فیت می‌کنیم
    target_encoder.fit(train_df[['PU_DO']], train_df[target])
    
    train_df['PU_DO_encoded'] = target_encoder.transform(train_df[['PU_DO']])
    val_df['PU_DO_encoded'] = target_encoder.transform(val_df[['PU_DO']])

    # ========================
    # فیچرهای نهایی برای مدل
    # ========================
    # PU_DO را حذف می‌کنیم و نسخه encode شده‌اش را نگه می‌داریم
    final_categorical = ['hour_category']                    # فقط این را وان‌هات می‌کنیم
    final_numerical = ['trip_distance', 'pickup_dayofweek', 'PU_DO_encoded']

    # ========================
    # وکتورایز کردن (فقط hour_category)
    # ========================
    dv = DictVectorizer()
    
    train_dicts = train_df[final_categorical + final_numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = val_df[final_categorical + final_numerical].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    y_train = train_df[target].values
    y_val = val_df[target].values

    print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")

    # ========================
    # آموزش مدل
    # ========================
    print("MODEL STARTING")
    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # پیش‌بینی
    y_pred = model.predict(X_val)

    # ========================
    # محاسبه متریک‌ها
    # ========================
    mae = mean_absolute_error(y_val, y_pred)
    rmse = mean_squared_error(y_val, y_pred)   # اصلاح شده
    r2 = r2_score(y_val, y_pred)

    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}, R2: {r2:.4f}")

    # ========================
    # لاگ کردن در MLflow
    # ========================
    mlflow.log_params({
        "model_type": "RandomForest",
        "n_estimators": 150,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "target_encoding": "PU_DO",
        "smoothing": 15,
        "min_samples_leaf_target": 30,
        "final_categorical": str(final_categorical),
        "final_numerical": str(final_numerical)
    })

    mlflow.log_metrics({
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
    model_name = "NYC_DURATION_PREDOCTION"
    # ========================
    # ذخیره مدل و Encoderها
    # ========================
    model_info = mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name,
            metadata={
                "training_data_months": "Jan-Feb 2026",
                "features": ["trip_distance", "PU_DO_encoded", "hour_category"],
                "target_encoding": True
            }
        )
    
    client = MlflowClient()
    client.update_model_version(
            name=model_name,
            version=model_info.registered_model_version,
            description="Trained with Target Encoding on PU_DO. Best model so far.1234567"
        )


    client.set_model_version_tag(
            name=model_name,
            version=model_info.registered_model_version,
            key="validation_rmse",
            value=str(round(rmse, 3))
        )

    # ذخیره TargetEncoder
    joblib.dump(target_encoder, "target_encoder.pkl")
    mlflow.log_artifact("target_encoder.pkl", artifact_path="preprocessing")

    # ذخیره DictVectorizer
    joblib.dump(dv, "dict_vectorizer.pkl")
    mlflow.log_artifact("dict_vectorizer.pkl", artifact_path="preprocessing")

    print("Run completed and logged to MLflow.")

2026/07/29 16:28:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train shape: (2000, 8), Validation shape: (500, 8)
MODEL STARTING
MAE: 1.931, RMSE: 13.211, R2: 0.9430


2026/07/29 16:28:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'NYC_DURATION_PREDOCTION' already exists. Creating a new version of this model...
2026/07/29 16:28:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: NYC_DURATION_PREDOCTION, version 6
Created version '6' of model 'NYC_DURATION_PREDOCTION'.


Run completed and logged to MLflow.
🏃 View run rf_target_encoding_v7 at: http://127.0.0.1:5000/#/experiments/1/runs/1c1ad0f7b9364cd3b0e087e27c1cd171
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [60]:
from mlflow.tracking import MlflowClient
client = MlflowClient("http://127.0.0.1:5000")

In [67]:
client.update_model_version(
            name="NYC_DURATION_PREDOCTION",
            version=model_info.registered_model_version,
            description="Trained with Target Encoding on PU_DO. Best model so far."
        )

NameError: name 'model_info' is not defined

client.search_registered_models()
mlflow.search_experiments()
client.search_runs(experiment_ids='1')[0].info.run_id
run_id = client.search_runs(experiment_ids='1')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name='NYC_Duration_Model'
)

# run the xgboost models with optuma 

In [60]:
import mlflow 
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment('nyc_duration_prediction_boosting_model')

Tracking URI: http://127.0.0.1:5000


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1785662749491, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785662749491, lifecycle_stage='active', name='nyc_duration_prediction_boosting_model', tags={}, trace_location=None, workspace='default'>

In [73]:
def select_features(df: pd.DataFrame):
    """Select features and define categorical vs numerical."""
    categorical = ['hour_category']           # فقط این را وکتورایز می‌کنیم (تعداد دسته کم است)
    
    numerical = [
        'trip_distance',
        'PULocationID',
        'DOLocationID',
        'pickup_dayofweek',
        'pickup_hour',
        'is_rush_hour',
        'is_weekend'
    ]
    target = 'duration'

    
    df = df[categorical + numerical + [target]].copy()
    return df, categorical, numerical, target


def vectorize_features(train_df, val_df, categorical, numerical):
    """Convert features using DictVectorizer."""
    print("Vectorizing features...")

    dv = DictVectorizer()

    train_dicts = train_df[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = val_df[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    return X_train, X_val, dv


In [62]:
import mlflow
import mlflow.sklearn
import joblib
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction import DictVectorizer
from pathlib import Path

# ========================
# انتخاب فیچرها
# ========================
print(train_df)
train_df, categorical, numerical, target = select_features(train_df)
val_df, _, _, _ = select_features(val_df)

# ========================
# Target Encoding برای PU_DO (حرفه‌ای)
# ========================
target_encoder = TargetEncoder(
    cols=['PU_DO'], 
    smoothing=15, 
    min_samples_leaf=30
)

# ========================
# وکتورایز کردن (فقط hour_category)
# ========================
dv= DictVectorizer()

train_dicts = train_df[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = val_df[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_train = train_df[target].values
y_val = val_df[target].values

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")


      VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0            2  2026-02-25 15:37:27   2026-02-25 15:49:43              1.0   
1            2  2026-01-11 02:16:38   2026-01-11 02:42:07              1.0   
2            2  2026-02-14 00:41:06   2026-02-14 00:44:53              1.0   
3            2  2026-02-02 14:02:25   2026-02-02 14:15:31              2.0   
4            1  2026-01-04 13:24:35   2026-01-04 13:42:08              1.0   
...        ...                  ...                   ...              ...   
4995         1  2026-01-08 11:47:59   2026-01-08 11:58:42              1.0   
4996         2  2026-01-02 00:39:28   2026-01-02 00:57:12              1.0   
4997         2  2026-01-30 20:29:32   2026-01-30 20:53:20              1.0   
4998         2  2026-01-31 09:19:27   2026-01-31 09:27:49              2.0   
4999         2  2026-01-16 11:44:16   2026-01-16 11:51:49              1.0   

      trip_distance  RatecodeID store_and_fwd_flag  PULocationI

In [91]:
import mlflow
import mlflow.sklearn
import joblib
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction import DictVectorizer
from pathlib import Path

# ========================
# انتخاب فیچرها
# ========================
print(train_df)
train_df, categorical, numerical, target = select_features(train_df)
val_df, _, _, _ = select_features(val_df)

# ========================
# Target Encoding برای PU_DO (حرفه‌ای)
# ========================
target_encoder = TargetEncoder(
    cols=['PU_DO'], 
    smoothing=15, 
    min_samples_leaf=30
)

# ========================
# وکتورایز کردن (فقط hour_category)
# ========================
dv= DictVectorizer()

train_dicts = train_df[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = val_df[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_train = train_df[target].values
y_val = val_df[target].values

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")


      VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0            2  2026-02-25 15:37:27   2026-02-25 15:49:43              1.0   
1            2  2026-01-11 02:16:38   2026-01-11 02:42:07              1.0   
2            2  2026-02-14 00:41:06   2026-02-14 00:44:53              1.0   
3            2  2026-02-02 14:02:25   2026-02-02 14:15:31              2.0   
4            1  2026-01-04 13:24:35   2026-01-04 13:42:08              1.0   
...        ...                  ...                   ...              ...   
4995         1  2026-01-08 11:47:59   2026-01-08 11:58:42              1.0   
4996         2  2026-01-02 00:39:28   2026-01-02 00:57:12              1.0   
4997         2  2026-01-30 20:29:32   2026-01-30 20:53:20              1.0   
4998         2  2026-01-31 09:19:27   2026-01-31 09:27:49              2.0   
4999         2  2026-01-16 11:44:16   2026-01-16 11:51:49              1.0   

      trip_distance  RatecodeID store_and_fwd_flag  PULocationI

In [75]:
import mlflow
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from mlflow.models import infer_signature


# ========================
# تابع هدف (Objective)
# ========================
def objective(trial, X_train, X_val, y_train, y_val):
    with mlflow.start_run(nested=True, run_name=f"lgb_trial_{trial.number}"):

        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        }

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)

        return rmse


# ========================
# تابع اصلی Tuning (بدون ثبت مدل)
# ========================
def run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50):
    
    with mlflow.start_run(run_name="lightgbm_optuna_tuning_version_4"):
        
        study = optuna.create_study(
            direction="minimize",
            study_name="LightGBM_Hyperparameter_Tuning",
            pruner=optuna.pruners.MedianPruner()
        )

        study.optimize(
            lambda trial: objective(trial, X_train, X_val, y_train, y_val),
            n_trials=n_trials,
            show_progress_bar=True
        )

        print("\n" + "="*60)
        print(f"بهترین RMSE: {study.best_value:.4f}")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")
        print("="*60)

        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_rmse", study.best_value)

        # ========================
        # آموزش مدل نهایی (فقط لاگ می‌شود، ثبت نمی‌شود)
        # ========================
        best_params = study.best_params.copy()
        best_params.update({
            'objective': 'regression',
            'metric': 'rmse',
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        })

        final_model = lgb.LGBMRegressor(**best_params)

        final_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        # ارزیابی نهایی
        y_pred = final_model.predict(X_val)
        final_rmse = root_mean_squared_error(y_val, y_pred)
        final_mae = mean_absolute_error(y_val, y_pred)
        final_r2 = r2_score(y_val, y_pred)

        mlflow.log_metrics({
            "final_rmse": final_rmse,
            "final_mae": final_mae,
            "final_r2": final_r2
        })

        # ========================
        # فقط مدل را لاگ می‌کنیم (ثبت در Registry نمی‌کنیم)
        # ========================
        signature = infer_signature(X_train, y_train)
        mlflow.sklearn.log_model(
            sk_model=final_model,
            artifact_path="model",
            signature=signature,                    
            skops_trusted_types=[
                "lightgbm.sklearn.LGBMRegressor",
                "lightgbm.basic.Booster",
                "collections.OrderedDict"
            ],
            metadata={
                "description": "this run use without auto registery model",
                "training_period": "AUG 2026",
                "feature_engineering": "PU_OP => target endcoding",
                "number_of_trials": 50,
                "best_rmse": round(final_rmse, 4),
            }
        )

        print("\nمدل نهایی آموزش دیده و لاگ شد (هنوز در Registry ثبت نشده).")
        return final_model, study

# ========================
# اجرا
# ========================
if __name__ == "__main__":
    run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50)

[I 2026-08-02 17:12:33,930] A new study created in memory with name: LightGBM_Hyperparameter_Tuning
  0%|                                                                                                         | 0/50 [00:00<?, ?it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 15.1023:   2%|█▏                                                           | 1/50 [00:00<00:05,  9.80it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:   4%|██▍            

🏃 View run lgb_trial_0 at: http://127.0.0.1:5000/#/experiments/2/runs/e32c1467d30f49568c46e30cd6da28e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:34,046] Trial 0 finished with value: 15.102300543142187 and parameters: {'n_estimators': 1139, 'num_leaves': 82, 'max_depth': 3, 'learning_rate': 0.10482032118852576, 'feature_fraction': 0.9314546953843585, 'bagging_fraction': 0.9729029620502969, 'bagging_freq': 1, 'min_child_samples': 29, 'reg_alpha': 1.6304501134457477e-05, 'reg_lambda': 0.03372378934919827}. Best is trial 0 with value: 15.102300543142187.
🏃 View run lgb_trial_1 at: http://127.0.0.1:5000/#/experiments/2/runs/8c0cddb5d4994ad0bad6a46a38a5bfa4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:34,148] Trial 1 finished with value: 15.084674165035253 and parameters: {'n_estimators': 1838, 'num_leaves': 114, 'max_depth': 10, 'learning_rate': 0.27581106659014887, 'feature_fraction': 0.8463702002959362, 'bagging_frac

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:   6%|███▋                                                         | 3/50 [00:00<00:05,  8.33it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:   6%|███▋                                                         | 3/50 [00:00<00:05,  8.33it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' i

🏃 View run lgb_trial_2 at: http://127.0.0.1:5000/#/experiments/2/runs/ecd8bdd715dd424a933cde6b090d7c6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:34,289] Trial 2 finished with value: 15.100999871729302 and parameters: {'n_estimators': 344, 'num_leaves': 94, 'max_depth': 12, 'learning_rate': 0.09533221769294067, 'feature_fraction': 0.6074162688797857, 'bagging_fraction': 0.7733909170167307, 'bagging_freq': 1, 'min_child_samples': 19, 'reg_alpha': 0.019261067420556107, 'reg_lambda': 0.0010173045578299891}. Best is trial 1 with value: 15.084674165035253.
🏃 View run lgb_trial_3 at: http://127.0.0.1:5000/#/experiments/2/runs/b48b1891affc4945a36018869417786e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:34,360] Trial 3 finished with value: 15.096647139659852 and parameters: {'n_estimators': 863, 'num_leaves': 70, 'max_depth': 9, 'learning_rate': 0.060614296527207345, 'feature_fraction': 0.9308449832250669, 'bagging_fracti

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:  10%|██████                                                       | 5/50 [00:00<00:04, 10.42it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:  14%|████████▌                                                    | 7/50 [00:00<00:03, 11.66it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' i

...

🏃 View run lgb_trial_44 at: http://127.0.0.1:5000/#/experiments/2/runs/8101c0189d7e4971becadeb8cf165ade
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:37,595] Trial 44 finished with value: 15.092606604142455 and parameters: {'n_estimators': 1992, 'num_leaves': 100, 'max_depth': 8, 'learning_rate': 0.20487928752022685, 'feature_fraction': 0.683997201417791, 'bagging_fraction': 0.645901385093022, 'bagging_freq': 4, 'min_child_samples': 10, 'reg_alpha': 0.06168359953297682, 'reg_lambda': 1.4569023785473102e-05}. Best is trial 1 with value: 15.084674165035253.
🏃 View run lgb_trial_45 at: http://127.0.0.1:5000/#/experiments/2/runs/0fe93eb11f1d4228b0cb56bcbbf29c66
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:37,695] Trial 45 finished with value: 15.099828504765899 and parameters: {'n_estimators': 1813, 'num_leaves': 136, 'max_depth': 10, 'learning_rate': 0.010481627999814601, 'feature_fraction': 0.761747076658735, 'bagging_f

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:  94%|████████████████████████████████████████████████████████▍   | 47/50 [00:03<00:00, 12.08it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 1. Best value: 15.0847:  98%|██████████████████████████████████████████████████████████▊ | 49/50 [00:03<00:00, 12.33it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' i

🏃 View run lgb_trial_47 at: http://127.0.0.1:5000/#/experiments/2/runs/5f15cb126a854e219c5bca8cfd611901
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:37,861] Trial 47 finished with value: 15.096256531521812 and parameters: {'n_estimators': 1684, 'num_leaves': 144, 'max_depth': 7, 'learning_rate': 0.07837649866979554, 'feature_fraction': 0.8852208223955267, 'bagging_fraction': 0.6954493920222342, 'bagging_freq': 5, 'min_child_samples': 78, 'reg_alpha': 4.0213884265649584e-08, 'reg_lambda': 0.00315691170734235}. Best is trial 1 with value: 15.084674165035253.
🏃 View run lgb_trial_48 at: http://127.0.0.1:5000/#/experiments/2/runs/2d8ec24557b1476dac6b5504f7966018
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:37,933] Trial 48 finished with value: 15.092692652527061 and parameters: {'n_estimators': 554, 'num_leaves': 117, 'max_depth': 4, 'learning_rate': 0.26075730107216155, 'feature_fraction': 0.9252414774980278, 'bagging_f

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
2026/08/02 17:12:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/02 17:12:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



مدل نهایی آموزش دیده و لاگ شد (هنوز در Registry ثبت نشده).
🏃 View run lightgbm_optuna_tuning_version_4 at: http://127.0.0.1:5000/#/experiments/2/runs/a6dfff346c6749dab4d7d6b6ab656c44
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [81]:
train_df, categorical, numerical, target = select_features(train_df)
val_df, _, _, _ = select_features(val_df)

# ========================
# Target Encoding برای PU_DO (حرفه‌ای)
# ========================
target_encoder = TargetEncoder(
    cols=['PU_DO'], 
    smoothing=15, 
    min_samples_leaf=30
)

# فقط روی داده Train فیت می‌کنیم
target_encoder.fit(train_df[['PU_DO']], train_df[target])

train_df['PU_DO_encoded'] = target_encoder.transform(train_df[['PU_DO']])
val_df['PU_DO_encoded'] = target_encoder.transform(val_df[['PU_DO']])

# ========================
# فیچرهای نهایی برای مدل
# ========================
# PU_DO را حذف می‌کنیم و نسخه encode شده‌اش را نگه می‌داریم
final_categorical = ['hour_category']                    # فقط این را وان‌هات می‌کنیم
final_numerical = ['trip_distance', 'pickup_dayofweek', 'PU_DO_encoded']

# ========================
# وکتورایز کردن (فقط hour_category)
# ========================
dv = DictVectorizer()

train_dicts = train_df[final_categorical + final_numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = val_df[final_categorical + final_numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_train = train_df[target].values
y_val = val_df[target].values

KeyError: "None of [Index(['PU_DO'], dtype='object')] are in the [columns]"

In [76]:
import mlflow
import optuna
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from mlflow.models import infer_signature


# ========================
# تابع هدف (Objective)
# ========================
def objective(trial, X_train, X_val, y_train, y_val):
    with mlflow.start_run(nested=True, run_name=f"lgb_trial_{trial.number}"):

        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        }

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)

        return rmse


# ========================
# تابع اصلی اجرای Tuning
# ========================
def run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50):
    
    with mlflow.start_run(run_name="lightgbm_optuna_tuning_version_7"):
        
        study = optuna.create_study(
            direction="minimize",
            study_name="LightGBM_Hyperparameter_Tuning",
            pruner=optuna.pruners.MedianPruner()
        )

        study.optimize(
            lambda trial: objective(trial, X_train, X_val, y_train, y_val),
            n_trials=n_trials,
            show_progress_bar=True
        )

        print("\n" + "="*60)
        print(f"بهترین RMSE: {study.best_value:.4f}")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")
        print("="*60)

        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_rmse", study.best_value)

        # ========================
        # آموزش مدل نهایی
        # ========================
        best_params = study.best_params.copy()
        best_params.update({
            'objective': 'regression',
            'metric': 'rmse',
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        })

        final_model = lgb.LGBMRegressor(**best_params)

        final_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        # ارزیابی نهایی
        y_pred = final_model.predict(X_val)
        final_rmse = root_mean_squared_error(y_val, y_pred)
        final_mae = mean_absolute_error(y_val, y_pred)
        final_r2 = r2_score(y_val, y_pred)

        mlflow.log_metrics({
            "final_rmse": final_rmse,
            "final_mae": final_mae,
            "final_r2": final_r2
        })

        # ========================
        # ذخیره Artifactهای پیش‌پردازش (خیلی مهم)
        # ========================
        # ذخیره TargetEncoder
        joblib.dump(target_encoder, "target_encoder.pkl")
        mlflow.log_artifact("target_encoder.pkl", artifact_path="preprocessing")

        # ذخیره DictVectorizer
        joblib.dump(dv, "dict_vectorizer.pkl")
        mlflow.log_artifact("dict_vectorizer.pkl", artifact_path="preprocessing")

        # ========================
        # لاگ کردن مدل (با Signature و Metadata)
        # ========================
        signature = infer_signature(X_train, y_train)

        mlflow.sklearn.log_model(
            sk_model=final_model,
            artifact_path="model",
            signature=signature,
            skops_trusted_types=[
                "lightgbm.sklearn.LGBMRegressor",
                "lightgbm.basic.Booster",
                "collections.OrderedDict"
            ],
            metadata={
                "description": "LightGBM model trained with Optuna (50 trials)",
                "training_period": "AUG 2026",
                "feature_engineering": "Target Encoding on PU_DO + One-Hot on hour_category",
                "number_of_trials": 50,
                "best_rmse": round(final_rmse, 4),
                "n_jobs": 2,
                "preprocessing_artifacts": "target_encoder.pkl, dict_vectorizer.pkl"
            }
        )
        print(f"نوع داده: {type(X_train)}")
        print(f"تعداد ردیف و ستون: {X_train.shape}")
        print("\n✅ مدل نهایی لاگ شد (همراه با Artifactهای پیش‌پردازش).")
        return final_model, study


# ========================
# اجرا
# ========================
if __name__ == "__main__":
    run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50)

[I 2026-08-02 17:12:52,427] A new study created in memory with name: LightGBM_Hyperparameter_Tuning
  0%|                                                                                                         | 0/50 [00:00<?, ?it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 15.0995:   0%|                                                                     | 0/50 [00:00<?, ?it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 15.0995:   4%|██▍            

🏃 View run lgb_trial_0 at: http://127.0.0.1:5000/#/experiments/2/runs/ea3c87d331ae4fcbb664cecea6e98ff2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:52,524] Trial 0 finished with value: 15.099502225620904 and parameters: {'n_estimators': 1297, 'num_leaves': 53, 'max_depth': 6, 'learning_rate': 0.012897712540701165, 'feature_fraction': 0.659401387654041, 'bagging_fraction': 0.6666794290038228, 'bagging_freq': 5, 'min_child_samples': 72, 'reg_alpha': 4.464984768264141, 'reg_lambda': 1.584346178662593e-07}. Best is trial 0 with value: 15.099502225620904.
🏃 View run lgb_trial_1 at: http://127.0.0.1:5000/#/experiments/2/runs/866892a0b49d410b98bcebf7607dc676
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:52,601] Trial 1 finished with value: 15.100063023686143 and parameters: {'n_estimators': 350, 'num_leaves': 20, 'max_depth': 12, 'learning_rate': 0.02511070734346076, 'feature_fraction': 0.8989495353475749, 'bagging_fraction'

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 15.0995:   8%|████▉                                                        | 4/50 [00:00<00:04, 10.30it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 15.0995:   8%|████▉                                                        | 4/50 [00:00<00:04, 10.30it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' i

🏃 View run lgb_trial_3 at: http://127.0.0.1:5000/#/experiments/2/runs/5d3b34403c3b4ffeba576f068244630a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:52,810] Trial 3 finished with value: 15.10003250180287 and parameters: {'n_estimators': 366, 'num_leaves': 45, 'max_depth': 7, 'learning_rate': 0.08408684924724112, 'feature_fraction': 0.9210109813510907, 'bagging_fraction': 0.9979115046760241, 'bagging_freq': 3, 'min_child_samples': 30, 'reg_alpha': 0.021857338130547754, 'reg_lambda': 1.089715640295601e-06}. Best is trial 0 with value: 15.099502225620904.
🏃 View run lgb_trial_4 at: http://127.0.0.1:5000/#/experiments/2/runs/ae1f9416fa1d4029ac360a5531cf1a32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:52,871] Trial 4 finished with value: 15.099625140583031 and parameters: {'n_estimators': 1522, 'num_leaves': 31, 'max_depth': 4, 'learning_rate': 0.1731553744374373, 'feature_fraction': 0.9078078563028591, 'bagging_fraction'

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 5. Best value: 15.0994:  12%|███████▎                                                     | 6/50 [00:00<00:03, 12.07it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 7. Best value: 15.0993:  16%|█████████▊                                                   | 8/50 [00:00<00:03, 12.47it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' i

...

🏃 View run lgb_trial_45 at: http://127.0.0.1:5000/#/experiments/2/runs/8e58d783a56d497e9ad89c9229ce72db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:56,101] Trial 45 finished with value: 15.08354499416038 and parameters: {'n_estimators': 764, 'num_leaves': 57, 'max_depth': 4, 'learning_rate': 0.2350970891305116, 'feature_fraction': 0.8088488074505947, 'bagging_fraction': 0.6235390288590413, 'bagging_freq': 6, 'min_child_samples': 53, 'reg_alpha': 0.025185967040489858, 'reg_lambda': 1.544092538268022e-05}. Best is trial 45 with value: 15.08354499416038.
🏃 View run lgb_trial_46 at: http://127.0.0.1:5000/#/experiments/2/runs/576760d214654855b302ba20665408e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:56,182] Trial 46 finished with value: 15.0868931397831 and parameters: {'n_estimators': 767, 'num_leaves': 59, 'max_depth': 4, 'learning_rate': 0.2984199973474693, 'feature_fraction': 0.8156623182155239, 'bagging_fraction'

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 45. Best value: 15.0835:  96%|████████████████████████████████████████████████████████▋  | 48/50 [00:03<00:00, 12.37it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 45. Best value: 15.0835: 100%|███████████████████████████████████████████████████████████| 50/50 [00:03<00:00, 12.56it/s]
/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' 

🏃 View run lgb_trial_48 at: http://127.0.0.1:5000/#/experiments/2/runs/2139b6b3b91847a5a607e00dbae1fc61
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:56,339] Trial 48 finished with value: 15.092224582451278 and parameters: {'n_estimators': 434, 'num_leaves': 74, 'max_depth': 5, 'learning_rate': 0.13131864493115705, 'feature_fraction': 0.8346683949051142, 'bagging_fraction': 0.6016323403349014, 'bagging_freq': 7, 'min_child_samples': 36, 'reg_alpha': 0.037614952824959524, 'reg_lambda': 0.00011526368964665746}. Best is trial 45 with value: 15.08354499416038.
🏃 View run lgb_trial_49 at: http://127.0.0.1:5000/#/experiments/2/runs/09991a1f22834e24b657255b8b84db72
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:12:56,410] Trial 49 finished with value: 15.088405809850846 and parameters: {'n_estimators': 710, 'num_leaves': 62, 'max_depth': 4, 'learning_rate': 0.1544376374021629, 'feature_fraction': 0.7951588121016119, 'bagging_frac

2026/08/02 17:13:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


نوع داده: <class 'scipy.sparse._csr.csr_matrix'>
تعداد ردیف و ستون: (5000, 5)

✅ مدل نهایی لاگ شد (همراه با Artifactهای پیش‌پردازش).
🏃 View run lightgbm_optuna_tuning_version_7 at: http://127.0.0.1:5000/#/experiments/2/runs/4212c04bd16445fcbfd16afe99ad5e41
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [77]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

model_name = "NYC_Taxi_Duration_LightGBM_version_7"

# ========================
# پیدا کردن بهترین Run
# ========================
runs = client.search_runs(
    experiment_ids=["2"],   # شماره Experiment خودت را بگذار
    filter_string="tags.mlflow.runName = 'lightgbm_optuna_tuning_version_7'",
    order_by=["metrics.best_rmse ASC"],
    max_results=1
)

if not runs:
    print("Run پیدا نشد!")
else:
    best_run = runs[0]
    run_id = best_run.info.run_id
    best_rmse = best_run.data.metrics.get("best_rmse")

    print(f"بهترین Run پیدا شد: {run_id}")
    print(f"بهترین RMSE: {best_rmse}")

    # ========================
    # ثبت مدل در Registry
    # ========================
    model_version = mlflow.register_model(
        model_uri=f"runs:/{run_id}/model",
        name=model_name
    )

    print(f"مدل ثبت شد → Version: {model_version.version}")

    # ========================
    # اضافه کردن توضیحات (Description)
    # ========================
    description = f"""
LightGBM model trained with Optuna hyperparameter tuning.

- Training Period: January & February 2026
- Best Validation RMSE: {round(best_rmse, 4)}
- Number of Optuna Trials: 50
- Feature Engineering: Target Encoding on PU_DO
- Final Features: trip_distance, pickup_dayofweek, PU_DO_encoded, hour_category (One-Hot)
- n_jobs: 2
"""

    client.update_model_version(
        name=model_name,
        version=model_version.version,
        description=description.strip()
    )

    # ========================
    # اضافه کردن تگ‌ها (Tags)
    # ========================
    tags = {
        "project": "NYC_Taxi_Duration",
        "team": "Data Science",
        "model_type": "LightGBM",
        "tuning_method": "Optuna",
        "training_period": "Jan-Feb 2026",
        "best_rmse": str(round(best_rmse, 4)),
        "number_of_trials": "50",
        "feature_engineering": "Target Encoding",
        "created_by": "Mohsen"
    }

    for key, value in tags.items():
        client.set_model_version_tag(
            name=model_name,
            version=model_version.version,
            key=key,
            value=value
        )

    # ========================
    # تنظیم Alias به عنوان Champion
    # ========================
    client.set_registered_model_alias(
        name=model_name,
        alias="champion",
        version=model_version.version
    )

    print(f"✅ مدل به عنوان @champion تنظیم شد (Version {model_version.version})")
    print("Description و Tags با موفقیت اضافه شدند.")

Registered model 'NYC_Taxi_Duration_LightGBM_version_7' already exists. Creating a new version of this model...
2026/08/02 17:13:01 WARNING mlflow.tracking._model_registry.fluent: Run with id f7fed88060994eedb5a032db48aac127 has no artifacts at artifact path 'model', registering model based on models:/m-12202b0219444afea77c586b05e991d7 instead
2026/08/02 17:13:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: NYC_Taxi_Duration_LightGBM_version_7, version 2


بهترین Run پیدا شد: f7fed88060994eedb5a032db48aac127
بهترین RMSE: 14.679830304492294
مدل ثبت شد → Version: 2


Created version '2' of model 'NYC_Taxi_Duration_LightGBM_version_7'.


✅ مدل به عنوان @champion تنظیم شد (Version 2)
Description و Tags با موفقیت اضافه شدند.


In [78]:
import pandas as pd
import joblib

print("=" * 70)
print("🔍 ساختار دقیق X_train (داده آموزش)")
print("=" * 70)

print(f"نوع داده: {type(X_train)}")
print(f"تعداد ردیف و ستون: {X_train.shape}")

# اگر X_train دیتافریم باشد
if hasattr(X_train, 'columns'):
    print("\nنام ستون‌ها:")
    print(X_train.columns.tolist())
    
    print("\n۵ ردیف اول:")
    print(X_train.head())

# اگر sparse matrix باشد (احتمالاً همین است)
else:
    print("\nX_train از نوع sparse matrix است.")
    
    # نمایش نام ستون‌ها از DictVectorizer
    if 'dv' in globals() and hasattr(dv, 'get_feature_names_out'):
        feature_names = dv.get_feature_names_out().tolist()
        print(f"\nنام ستون‌ها ({len(feature_names)} تا):")
        print(feature_names)
        
        # ذخیره برای استفاده بعدی
        joblib.dump(feature_names, "feature_names.pkl")
        print("\n✅ لیست ستون‌ها در feature_names.pkl ذخیره شد.")
    else:
        print("dv پیدا نشد.")

print("\n" + "=" * 70)

🔍 ساختار دقیق X_train (داده آموزش)
نوع داده: <class 'scipy.sparse._csr.csr_matrix'>
تعداد ردیف و ستون: (5000, 5)

X_train از نوع sparse matrix است.

نام ستون‌ها (5 تا):
['hour_category=evening_peak', 'hour_category=late_night', 'hour_category=midday', 'hour_category=morning_peak', 'hour_category=night']

✅ لیست ستون‌ها در feature_names.pkl ذخیره شد.



In [66]:
import mlflow
import pandas as pd
import joblib
from scipy.sparse import hstack

mlflow.set_tracking_uri("http://127.0.0.1:5000")

# ========================
# لود مدل و پیش‌پردازش
# ========================
model = mlflow.sklearn.load_model("models:/NYC_Taxi_Duration_LightGBM@champion")
target_encoder = joblib.load("target_encoder.pkl")
dv = joblib.load("dict_vectorizer.pkl")

print("✅ مدل و پیش‌پردازش‌ها لود شدند.")

# ========================
# داده نمونه
# ========================
df_real = pd.DataFrame({
    'hour_category': ['midday', 'late_night', 'late_night', 'midday', 'midday'],
    'trip_distance': [1.41, 5.49, 0.48, 0.66, 1.50],
    'PULocationID': [142, 148, 230, 162, 163],
    'DOLocationID': [161, 188, 162, 100, 186],
    'pickup_dayofweek': [2, 6, 5, 0, 6]
})

print("\n" + "="*60)
print("داده خام:")
print(df_real)
print("="*60)

# ========================
# مرحله ۱: ایجاد PU_DO + Target Encoding
# ========================
df_real['PU_DO'] = df_real['PULocationID'].astype(str) + '_' + df_real['DOLocationID'].astype(str)
df_real['PU_DO_encoded'] = target_encoder.transform(df_real[['PU_DO']])

print("\nبعد از Target Encoding:")
print(f"تعداد ستون df_real: {df_real.shape[1]}")
print(df_real.columns.tolist())

# ========================
# مرحله ۲: تبدیل hour_category به One-Hot
# ========================
cat_dicts = df_real[['hour_category']].to_dict(orient='records')
print(cat_dicts)
X_cat = dv.transform(cat_dicts)
X_cat_df = pd.DataFrame(X_cat.toarray(), columns=dv.get_feature_names_out())

print("\nبعد از One-Hot hour_category:")
print(f"تعداد ستون X_cat_df: {X_cat_df.shape[1]}")
print("ستون‌های X_cat_df:", X_cat_df.columns.tolist())

# ========================
# مرحله ۳: ساخت X_new با ترتیب درست (مهم!)
# ========================
# ترتیب صحیح (بر اساس خروجی قبلی X_train):
# ['PU_DO_encoded', hour cats (5 ستون), 'pickup_dayofweek', 'trip_distance']

pu_do_part = df_real[['PU_DO_encoded']].values
cat_part   = X_cat
day_part   = df_real[['pickup_dayofweek']].values
dist_part  = df_real[['trip_distance']].values

X_new = X_cat_df

print("\n" + "="*60)
print(f"تعداد فیچر نهایی X_new: {X_new.shape[1]}")
print("="*60)

# ========================
# پیش‌بینی
# ========================
predictions = model.predict(X_new)

df_real['predicted_duration'] = predictions.round(2)

print("\nنتایج پیش‌بینی:")
print(df_real[['trip_distance', 'PULocationID', 'DOLocationID', 'hour_category', 'predicted_duration']])

✅ مدل و پیش‌پردازش‌ها لود شدند.

داده خام:
  hour_category  trip_distance  PULocationID  DOLocationID  pickup_dayofweek
0        midday           1.41           142           161                 2
1    late_night           5.49           148           188                 6
2    late_night           0.48           230           162                 5
3        midday           0.66           162           100                 0
4        midday           1.50           163           186                 6


NotFittedError: Must train encoder before it can be used to transform data.

In [112]:
print("\nتعداد فیچر X_train:", X_train.shape[1])
print("تعداد فیچر X_new   :", X_new.shape[1])
print("ستون‌ها یکسان هستند؟", X_train.columns.tolist() == X_new.columns.tolist())


تعداد فیچر X_train: 8
تعداد فیچر X_new   : 11


AttributeError: 'csr_matrix' object has no attribute 'columns'